# Week 12 — Python Scripts, Testing, and GitHub

**COMP 1110 — Introduction to Computer Programming**

**Tuesday, November 24 & Thursday, November 26, 2026**
9:30 – 11:20 AM · Room OM 1241

Instructor: **Quan Nguyen**

*Two sessions: scripts and `pytest` (Tue) · GitHub as a team (Thu).*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TRU-CS/COMP_1110/blob/main/COMP_1110/notebooks/week12_scripts_testing_github.ipynb)

*Run this notebook yourself: press the power button at the top of the page to
execute cells right here, or click the badge to open an editable copy in Google
Colab (a Google account is required).*


## Where we are

- **Last week (Week 11):** tuples, sets — and Midterm 2. Every Python tool your
  project needs, you now have.
- **This week (Week 12):** leaving the notebook. Running your project as a real
  **script**, running its **tests** from the terminal, and sharing it through **GitHub**.
- **Next week (Week 13):** PEP 8 and code style, then project presentations.

> **Change of plan.** This week was going to be classes and objects. It is now the
> week your project actually gets built. A class is **no longer required** in the
> final project — you may still write one if it helps.

## Today's outcomes

By the end of these two classes you will be able to:

1. Run a Python file from the terminal, and explain how a script differs from a
   notebook. **[CLO 1]**
2. Lay out `project.py` as `main()` + functions + the `if __name__ == "__main__":`
   guard, and explain what the guard is for. **[CLO 2]**
3. Run `test_project.py` with `pytest`, and read a failure report to find the bug.
   **[CLO 4]**
4. Clone your team repository, then `pull` → `commit` → `push` your own work and
   repair a merge conflict. **[CLO 5]**

## Reading

- [Final project brief](../final_project.md) — §1 *Required files* and *Project
  layout*. Read it again tonight; it is what gets marked.
- [Get Started — pytest](https://docs.pytest.org/en/stable/getting-started.html) —
  installing and running pytest (from Week 8, now for real).
- [GitHub — Hello World](https://docs.github.com/en/get-started/start-your-journey/hello-world)
  — from Week 6, if commits and pushes feel shaky.

Bring a **laptop with Python, Git, and VS Code** to both classes. Thursday is
hands-on in your team repository.

# Part 1 — From notebook to script

## A notebook is not a program

So far you have run Python in **cells**: run one, edit another, run them in any order.

A **script** is one plain `.py` file that Python runs **top to bottom, from a fresh
start, every time**.

| | Notebook (`.ipynb`) | Script (`.py`) |
| :--- | :--- | :--- |
| Runs | one cell at a time, any order | the whole file, top to bottom |
| Remembers | variables from earlier cells | nothing — fresh every run |
| `input()` | awkward | works normally |
| Who can run it | someone with Jupyter / Colab | anyone with Python |

Your project is marked by running `python3 project.py`. **It has to work as a script.**

## The terminal — three commands

The **terminal** is a window where you type commands instead of clicking.
In VS Code: **File → Open Folder…** (your project folder), then
**Terminal → New Terminal**. It opens *already inside that folder*.

```text
pwd          where am I?             (Windows PowerShell: pwd works too)
ls           what files are here?    (Windows: ls or dir)
cd gradebook move into a folder      (cd .. goes back up one)
```

> **Windows:** where these notes say `python3`, type `python` (or `py`).
> Everything else is the same.

## Your first script

Save this in VS Code as `hello.py`:

```python
name = input("What is your name? ")
print(f"Hello, {name}! This came from a script.")
```

Then, in the terminal, in the same folder:

```text
$ python3 hello.py
What is your name? Amina
Hello, Amina! This came from a script.
$
```

`$` is the prompt — you do not type it. The program runs, ends, and hands the
terminal back. Run it again and it starts from nothing.

### Predict the output (1)

`gradebook.py` contains exactly this. What appears when you run `python3 gradebook.py`?

```python
def main():
    print("Welcome to the gradebook")


print("Loading...")
```

**Answer: only `Loading...`**

`def main():` **defines** the function — it does not run it. Nothing in the file
ever **calls** `main()`, so "Welcome" is never printed.

In a notebook you would have called it in the next cell. In a script, *you* have to
put the call in the file. That is the last line of every project file.

## The shape of `project.py`

```python
def main():
    """Ask for scores and report the average."""
    ...                      # input() and print() live here


def average(scores):
    """Return the mean of a list of scores."""
    return sum(scores) / len(scores)


if __name__ == "__main__":
    main()
```

Three parts, always in this order: **`main`**, **your functions**, **the guard**.
The guard is the strange bit. To see why it is there, we need `__name__`.

## `__name__` — which file is "the main one"?

Python gives every file a hidden variable, `__name__`:

- the file you **run** gets `__name__ == "__main__"`;
- a file that is **imported** gets its own name — `tools`, `project`, …

A notebook counts as the file being run:

In [1]:
print(__name__)

__main__


### Watch it change on import

This cell writes a small file, `tools.py`, so we can import it.

In [2]:
tools_code = """print("tools.py is running; __name__ is", __name__)


def shout(text):
    \"\"\"Return text in capitals with an exclamation mark.\"\"\"
    return text.upper() + "!"


if __name__ == "__main__":
    print(shout("hello"))
"""

with open("tools.py", "w") as handle:
    handle.write(tools_code)

In [3]:
import tools

print(tools.shout("hi"))

tools.py is running; __name__ is tools
HI!


Imported, `tools.py` ran its top-level `print` — with `__name__` set to `tools` —
so the guarded `print(shout("hello"))` was **skipped**. Its function was still
defined and usable.

### Predict the output (2)

`tools.py` is the file above. `app.py`, in the same folder, is:

```python
import tools

print(tools.shout("go"))
```

What does `python3 app.py` print? And what does `python3 tools.py` print?

**`python3 app.py`:**

```text
tools.py is running; __name__ is tools
GO!
```

**`python3 tools.py`:**

```text
tools.py is running; __name__ is __main__
HELLO!
```

Same file, two behaviours: the guard runs `main` code **only when the file is the
one being run**. Imported, it just provides its functions.

## Why your project needs the guard

`test_project.py` starts with `from project import average`. That **imports**
`project.py` — and importing runs the file.

Without the guard, the last line of `project.py` is a bare `main()`. pytest imports
it, `main()` calls `input()`, and pytest stops before a single test runs:

```text
ERROR collecting test_project.py
project.py:17: in <module>
    main()
project.py:2: in main
    score = float(input("Score: "))
E   OSError: pytest: reading from stdin while output is captured!  Consider using `-s`.
```

**Do not** add `-s`. Add the guard.

## Keep `input()` and `print()` in `main`

pytest cannot type into `input()`, and it does not read what you `print()`.
So split the work:

| `main()` | your other functions |
| :--- | :--- |
| talks to the user: `input()`, `print()` | take **parameters**, **return** a value |
| hard to test automatically | easy to test: call, then `assert` |

```python
def main():
    raw = input("Scores, separated by commas: ")
    print(f"Average: {average(parse_scores(raw)):.1f}")


def parse_scores(text):
    """Turn "70, 85,90" into [70.0, 85.0, 90.0]."""
    scores = []
    for piece in text.split(","):
        scores.append(float(piece))
    return scores
```

Every function you count toward your nine should look like `parse_scores`.

In [4]:
# Live coding 1 — notebook cell -> script + test
#
# A teammate wrote this in a notebook cell:
#
#     marks = input("Marks: ").split()
#     total = 0
#     for mark in marks:
#         total = total + int(mark)
#     print("Average:", total / len(marks))
#
# Together: turn it into a script, marks.py, with
#   - average_mark(marks): takes a list of numbers, returns the average
#     (and returns 0 for an empty list instead of crashing)
#   - main(): does the input() and print()
#   - the __name__ guard
# then write test_average_mark() and run it.

In [5]:
# Solution
marks_code = """def main():
    \"\"\"Ask for marks on one line and print their average.\"\"\"
    marks = []
    for piece in input("Marks: ").split():
        marks.append(int(piece))
    print("Average:", average_mark(marks))


def average_mark(marks):
    \"\"\"Return the average of marks, or 0 if there are none.\"\"\"
    if len(marks) == 0:
        return 0
    return sum(marks) / len(marks)


if __name__ == "__main__":
    main()
"""
with open("marks.py", "w") as handle:
    handle.write(marks_code)

from marks import average_mark   # no input() appears: the guard works


def test_average_mark():
    assert average_mark([70, 80, 90]) == 80
    assert average_mark([55]) == 55
    assert average_mark([]) == 0


test_average_mark()
print("test_average_mark passed")

test_average_mark passed


## Common errors — running scripts

| You see | What happened | Fix |
| :--- | :--- | :--- |
| `can't open file '.../hello.py': [Errno 2] No such file or directory` | The terminal is in a different folder from the file. | `ls` to look, `cd` into the right folder. |
| `'python3' is not recognized…` (Windows) | Windows calls it `python` or `py`. | `python hello.py` |
| `SyntaxError` on `python3 hello.py` | You typed it at the `>>>` Python prompt, not the terminal. | `exit()`, then run it again. |
| The script runs and prints **nothing** | `main` is defined but never called. | Add the guard + `main()` at the bottom. |

If the terminal looks wrong, `pwd` and `ls` answer most questions.

# Part 2 — The testing script

## Two files, one folder

Week 8 showed the layout. Today you run it.

```text
gradebook/
├── project.py          main() + your functions + the guard
└── test_project.py     from project import ...  +  def test_...():
```

```python
# test_project.py
from project import letter_grade


def test_letter_grade():
    assert letter_grade(85) == "A"
    assert letter_grade(80) == "A"     # the boundary
    assert letter_grade(49) == "F"
```

Each function you write gets **one test function named after it**: `letter_grade`
→ `test_letter_grade`. That rule is in the project brief.

## Install pytest — once per computer

pytest is not part of Python; install it with **pip**, Python's installer:

```text
$ python3 -m pip install pytest
...
Successfully installed pytest-9.2.0 ...
```

Then, from the folder that holds both files:

```text
$ python3 -m pytest
============================= test session starts ==============================
collected 1 item

test_project.py .                                                        [100%]

============================== 1 passed in 0.01s ===============================
```

Why `python3 -m pytest` and not just `pytest`? It uses **the same Python** that
runs your script, so it finds the same files. On Windows: `py -m pytest`.

## Reading a failure report

Someone "simplifies" `score >= 80` to `score > 80`. Run the tests again:

```text
test_project.py F                                                        [100%]

______________________________ test_letter_grade _______________________________

    def test_letter_grade():
        assert letter_grade(85) == "A"
>       assert letter_grade(80) == "A"
E       AssertionError: assert 'B' == 'A'

test_project.py:6: AssertionError
FAILED test_project.py::test_letter_grade - AssertionError: assert 'B' == 'A'
============================== 1 failed in 0.01s ===============================
```

- **`F`** — one failing test. **`>`** — the exact line that failed.
- **`assert 'B' == 'A'`** — the function **returned `'B'`**, the test expected `'A'`.
- `test_project.py:6` — file and line. Go there, then go read `letter_grade`.

### Predict the output (3)

```python
# project.py
def is_passing(score):
    """Return True if score is a pass (50 or more)."""
    return score > 50
```

```python
# test_project.py
from project import is_passing


def test_is_passing():
    assert is_passing(75)
    assert is_passing(50)


def test_is_failing():
    assert not is_passing(20)


def check_zero():
    assert not is_passing(0)
```

You run `python3 -m pytest`. How many tests are **collected**? Which pass?

**`collected 2 items`**, shown as **`F.`** → `1 failed, 1 passed`.

- `check_zero` does not start with `test_`, so pytest **never sees it**.
- `test_is_passing` fails on `is_passing(50)`: `50 > 50` is `False`. The docstring
  says 50 passes — the code is wrong, and the test found it.
- `test_is_failing` passes.

A test is only a test if pytest can find it **and** it says `assert`.

## What makes a test worth having

For each function, test:

1. **A typical value** — `average([70, 80, 90]) == 80`
2. **A boundary** — exactly 50, exactly 80, a list of one item
3. **An awkward input** — an empty list, `"abc"` where a number should be, zero

```python
def test_parse_scores():
    assert parse_scores("70, 85,90") == [70.0, 85.0, 90.0]   # typical
    assert parse_scores("100") == [100.0]                    # one item
```

> **A test that cannot fail is not a test.**
> `def test_average(): average([1, 2])` has no `assert` — it passes even
> when `average` returns nonsense.

## Common errors — running pytest

| You see | What happened | Fix |
| :--- | :--- | :--- |
| `No module named pytest` | pytest is not installed for *this* Python. | `python3 -m pip install pytest` |
| `ERROR: file or directory not found: test_project.py` | Wrong folder. | `cd` to the folder with both files. |
| `collected 0 items` · `no tests ran` | Nothing is named `test_…`. | Rename file **and** functions `test_…`. |
| `OSError: pytest: reading from stdin…` | Importing `project.py` ran `main()`. | Add the `__name__` guard. |
| `ModuleNotFoundError: No module named 'project'` | The test file is not next to `project.py`, or the import is misspelled. | Same folder, `from project import …` |

Zero tests passing is **not** the same as your code working.

## End of Tuesday — before Thursday

On **your laptop**, before Thursday's class:

1. `python3 --version` and `git --version` both print a version number.
2. `python3 -m pip install pytest`, then `python3 -m pytest --version` works.
3. You can sign in to GitHub and see your **team repository** in the browser.
4. You know which **three functions** you own (they are in your proposal).

Stuck on any of these? Come at 9:15 on Thursday, or post on Moodle tonight.

# Part 3 — GitHub as a team (Thursday)

## Recap: the loop from Week 6

```text
git pull                         get your teammates' latest work
  ... edit, run python3 project.py, run python3 -m pytest ...
git status                       what have I changed?
git add project.py test_project.py
git commit -m "Add parse_scores and its test"
git pull                         again: did anyone push while I worked?
git push                         send it to GitHub
```

**Repository** — the folder + its full history. **Commit** — one saved snapshot,
with your name on it. **Push / pull** — send to / bring from GitHub.

Today: doing this for real, in **your team's** repository, from **your** account.

## Step 1 — get the repository onto your laptop

On GitHub, open your team repository → green **Code** button → copy the
**HTTPS** URL. Then:

```text
$ cd Documents                      somewhere you will find it again
$ git clone https://github.com/your-team/comp1110-gradebook.git
$ cd comp1110-gradebook
$ code .                            open this folder in VS Code
```

**Once per laptop**, so your commits carry *your* name, and so `git pull` never
has to ask how to combine work:

```text
git config --global user.name "Amina Chen"
git config --global user.email "the-email-on-your-github-account@example.com"
git config --global pull.rebase false
```

## Step 2 — signing in so you can push

The first `git push` asks who you are. GitHub **does not accept your password**
here any more:

```text
remote: Support for password authentication was removed on August 13, 2021.
fatal: Authentication failed for 'https://github.com/your-team/...'
```

Use one of these instead:

- **Windows (Git for Windows):** a browser window opens — sign in there. Done.
- **Any computer, in VS Code:** push from the **Source Control** panel; it offers
  *Sign in with GitHub* in the browser.
- **Otherwise:** create a *personal access token* on GitHub (Settings → Developer
  settings) and paste it where Git asks for a password.

Got `403` / `Permission … denied to`? You have not **accepted the collaborator
invitation** yet — check your email.

## What does *not* go in the repository

Running pytest leaves clutter behind:

```text
$ git status
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	__pycache__/
	project.py
	test_project.py
```

`__pycache__/` is **generated** — Python rebuilds it whenever it likes. Never commit it.
(pytest's own `.pytest_cache/` folder hides itself, so you will not see that one.)

A file called **`.gitignore`** lists what Git should ignore. If your team picked
the *Python* template when creating the repo, you already have one. If not, create
`.gitignore` at the top level containing:

```text
__pycache__/
```

### Predict the output (4)

The repository has a `.gitignore` containing `__pycache__/`. Ben edits
`project.py`, runs `python3 -m pytest` (which creates `__pycache__/`), and creates a
new file `notes.txt`. What does `git status` list?

```text
Changes not staged for commit:
	modified:   project.py

Untracked files:
	notes.txt
```

- `project.py` is **modified** — Git already tracks it.
- `notes.txt` is **untracked** — new, and not ignored. Git will not commit it until
  someone `git add`s it.
- `__pycache__/` **does not appear at all**. That is what `.gitignore` does.

## Three people, one `project.py`

Your project must have all nine functions **in `project.py`**. Three people will be
editing one file. That is normal — Git merges it for you, **if**:

- you **`git pull` before you start**, and again before you push;
- you **commit small**: one function + its test per commit;
- you **don't reformat or reorder code you don't own**;
- you push **green**: `python3 -m pytest` passes before every push.

When two people change **the same lines** — for example both add a new function in
the same spot — Git cannot decide, and asks you. That is a **merge conflict**.

## A merge conflict, for real

Amina pushed `average`. Ben, meanwhile, committed `highest` in the same spot:

```text
$ git push
 ! [rejected]        main -> main (fetch first)
$ git pull
CONFLICT (content): Merge conflict in project.py
Automatic merge failed; fix conflicts and then commit the result.
```

`project.py` now contains **both** versions, fenced by markers:

```text
<<<<<<< HEAD
def highest(scores):
    """Return the largest score."""
    return max(scores)
=======
def average(scores):
    """Return the mean of scores."""
    return sum(scores) / len(scores)
>>>>>>> 36ce437bd9c0c40196d95eb1186dccf8346fe236
```

### Repairing it — four steps

1. **Decide.** Here we want **both** functions. (`HEAD` is *your* side; below
   `=======` is what you pulled.)
2. **Edit** the file: delete the three marker lines, keep both functions, put two
   blank lines between them. VS Code's *Accept Both Changes* button does this.
3. **Prove it:** `python3 -m pytest` and `python3 project.py` still work.
4. **Finish the merge:**

```text
git add project.py
git commit -m "Merge Amina's average with highest"
git push
```

A conflict is not a mistake — it is Git refusing to guess. It only becomes a
mistake if you push the `<<<<<<<` lines.

## Your commit history is your evidence

At the end of term I read three things side by side:

1. the **ownership table** in your proposal,
2. the **commit history** on GitHub,
3. the **peer assessments**.

So:

- Commit and push **from your own account** — not through a teammate's laptop.
- Check your commits show **your avatar and username** on GitHub. If they say
  someone unknown, your `user.email` does not match your GitHub account.
- Write messages that say **what** changed: `Add parse_scores and its test`, not
  `update` or `stuff`.

In [6]:
# Live coding 2 — tests that catch a bug
#
# A teammate wrote this and says "it works, I tried it":
#
#     def count_passing(marks):
#         """Return how many marks are a pass (50 or more)."""
#         count = 0
#         for mark in marks:
#             if mark > 50:
#                 count = count + 1
#         return count
#
# Together: write test_count_passing() with a typical case, a boundary,
# and an empty list. Which assert catches the bug? Then fix the function.

In [7]:
# Solution
def count_passing(marks):
    """Return how many marks are a pass (50 or more)."""
    count = 0
    for mark in marks:
        if mark >= 50:            # was: mark > 50
            count = count + 1
    return count


def test_count_passing():
    assert count_passing([80, 30, 65]) == 2     # typical
    assert count_passing([50, 49]) == 1         # boundary: caught the bug
    assert count_passing([]) == 0               # empty


test_count_passing()
print("test_count_passing passed")

test_count_passing passed


## Common errors — Git as a team

| You see | What happened | Fix |
| :--- | :--- | :--- |
| `! [rejected] main -> main (fetch first)` | A teammate pushed first. | `git pull`, then `git push`. |
| `fatal: Need to specify how to reconcile divergent branches` | Git wants to know how to combine work. | `git config --global pull.rebase false`, pull again. |
| `CONFLICT (content): Merge conflict in project.py` | Same lines changed on both sides. | Edit out the markers, test, `add`, `commit`, `push`. |
| A strange full-screen editor appears after `git pull` | Git wants a message for the merge. | In *vim*: type `:wq` then Enter. In *nano*: Ctrl+X. |
| `fatal: not a git repository` | Wrong folder. | `cd` into the cloned repository. |
| `nothing to commit, working tree clean` | You did not save, or edited a copy elsewhere. | Save the file; check `pwd`. |

## Summary

- A **script** runs top to bottom from a fresh start: `python3 project.py`
  (Windows: `python`). Run it from the folder it lives in.
- `project.py` = **`main()`** + **your functions** + **`if __name__ == "__main__": main()`**.
  The guard stops `main()` running when the file is imported — by pytest.
- Keep `input()`/`print()` in `main`; every other function takes parameters and
  **returns**, so it can be tested.
- `python3 -m pytest` runs every `test_…` function; read `>` and `E` lines to find
  the failing assert and what the function returned.
- Team loop: **pull → edit → test → commit → pull → push**, from your own
  account. A conflict is solved by editing the file, not by panicking.

## This week's worksheet — W11: Team Repo Lab

Done **in your team's GitHub repository**, on your laptop — not on TRU-Think.

- **Part A** — tools check, clone, your own `practice/<your-username>/` folder.
- **Part B** — *Bill Splitter*: a real script with `main()`, the guard, and `input()`.
- **Part C** — its testing script, run with `pytest`; break it on purpose and read
  the report.
- **Part D** — **one of your three project functions + its test**, pushed to the real
  `project.py` / `test_project.py`.
- **Trace and explain** — answers in your `RESULTS.md`.

**Released Tuesday Nov 24 · due Thursday Dec 3, 11:59 PM.** Nothing to upload: I read
your commits. Your best 10 of 11 worksheets count.

## Next class — Week 13, Tuesday Dec 1

**Formatting and code style:** PEP 8, naming, layout, auto-formatters and linters —
then the project polish checklist.

Reading: [PEP 8 — Style Guide for Python Code](https://peps.python.org/pep-0008/).

Before then: **every member** has at least one function and its test pushed to
your team's `project.py` from their own account.